In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*See the [API reference](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Функції Qiskit є експериментальною можливістю, доступною лише користувачам IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у статусі попереднього випуску і можуть змінюватися.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>
## Огляд
У квантовій хімії задача електронної структури полягає у знаходженні розв'язків електронного рівняння Шрьодінґера — квантових хвильових функцій, що описують поведінку електронів системи. Ці хвильові функції є векторами комплексних амплітуд, де кожна амплітуда відповідає внеску можливої конфігурації електронів.

Основний стан — це хвильова функція системи з найнижчою енергією, яка має особливе значення при вивченні молекулярних систем. Найточніший підхід до обчислення основного стану враховує всі можливі конфігурації електронів, але він стає неможливим для більших систем, оскільки кількість конфігурацій зростає експоненційно зі збільшенням розміру системи.

Ітераційний варіаційний квантовий власний солвер з передачею (HI-VQE) — це інноваційний гібридний квантово-класичний метод для точного оцінювання основного стану молекулярних систем. Він інтегрує квантове апаратне забезпечення з класичними обчисленнями, використовуючи квантові процесори для ефективного дослідження кандидатних конфігурацій електронів та обчислення відповідної хвильової функції на класичних комп'ютерах. Генеруючи компактні, але хімічно точні хвильові функції, HI-VQE сприяє дослідженням і відкриттям у квантовій хімії та матеріалознавстві.

![Зображення, що показує огляд алгоритму HI-VQE від Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE зменшує обчислювальну складність задачі електронної структури, ефективно оцінюючи основний стан з високою точністю. Він зосереджується на ретельно відібраній підмножині найбільш релевантних конфігурацій електронів, оптимізуючи як точність, так і ефективність.

Поєднуючи переваги класичних і квантових комп'ютерів, HI-VQE ітераційно вдосконалює й покращує поточну оцінку хвильової функції. Його унікальні техніки побудови підпростору допомагають зробити вибір конфігурацій ефективнішим, надаючи користувачам більший обчислювальний контроль та покращену точність у квантово-хімічних симуляціях.

Якщо хочеш детальніше дізнатися про алгоритм, можеш [прочитати відповідну дослідницьку статтю](https://arxiv.org/abs/2503.06292).
## Опис
Кількість конфігурацій електронів для молекулярної системи зростає експоненційно зі збільшенням розміру системи. Однак для певних електронних станів, таких як основний стан, зазвичай лише невелика частка конфігурацій суттєво впливає на енергію стану. Методи вибраної конфігураційної взаємодії (SCI) використовують цю розрідженість для зменшення обчислювальних витрат, ідентифікуючи та зосереджуючись на найбільш релевантних конфігураціях. Ця підмножина конфігурацій називається підпростором.

HI-VQE використовує притаманну квантовим комп'ютерам ефективність при поданні молекулярних систем для допомоги в пошуку підпростору. Він інтегрує класичні та квантові підпрограми для розв'язання задачі електронної структури з високою точністю. На відміну від існуючих квантових SCI-методів, HI-VQE поєднує варіаційне навчання, ітераційну побудову підпростору та попереднє діагоналізаційне відсіювання конфігурацій для підвищення ефективності шляхом зменшення кількості квантових вимірювань, ітерацій і витрат на класичну діагоналізацію. Тому HI-VQE можна застосовувати до більших молекулярних систем, які вимагають більше кубітів, і зменшує вартість розв'язання задачі заданого розміру до однакового ступеня точності.

![Зображення з детальним описом кожного кроку в алгоритмі HI-VQE від Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Для обчислення основного стану системи HI-VQE спочатку використовує класичний хімічний пакет PySCF для генерації молекулярного представлення з наданих користувачем вхідних даних, таких як молекулярна геометрія та інша молекулярна інформація. Потім він входить у гібридний квантово-класичний цикл оптимізації, ітераційно вдосконалюючи підпростір для оптимального представлення основного стану, мінімізуючи кількість включених конфігурацій. Цикл продовжується до виконання критеріїв збіжності, таких як розмір підпростору або стабільність енергії, після чого виводяться обчислена хвильова функція основного стану та енергія. Ці результати можна використовувати для побудови точних поверхонь потенційної енергії та проведення подальшого аналізу системи.

Цикл оптимізації зосереджується на налаштуванні параметрів квантової схеми для генерації якісного підпростору. HI-VQE пропонує три варіанти квантових схем: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) та [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Оптимізація ініціалізується поблизу референсного стану Гартрі–Фока завдяки його загальній придатності. Потім схема виконується на квантовому пристрої, і конфігурації вибираються з отриманого квантового стану перед поверненням у вигляді бінарних рядків. Через шуми квантового пристрою деякі відібрані конфігурації можуть бути фізично недійсними, не зберігаючи кількість електронів або спін. HI-VQE вирішує це за допомогою процесу відновлення конфігурацій з пакету [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), щоб користувачі могли або виправити недійсні конфігурації, або відкинути їх.

Дійсні конфігурації потім проходять необов'язковий крок відсіювання для видалення тих, що, за прогнозами, дають мінімальний внесок. Це зменшує розмірність підпростору, тим самим знижуючи вартість кроку діагоналізації. Якщо відсіювання увімкнено, то будується попередній підпросторовий гамільтоніан із дійсних конфігурацій і виконується діагоналізація з дуже м'якими критеріями завершення. Хоча точність отриманих амплітуд для кожної конфігурації є низькою, це ефективно для прогнозування, які конфігурації слід виключити з підпростору на даній ітерації, і обчислюється швидко.

Вибрані конфігурації додаються до підпростору, і гамільтоніан системи проєктується у цей підпростір. Підпростір оновлюється ітераційно, зберігаючи найбільш релевантні конфігурації між ітераціями. Цей підхід відрізняється від альтернативних методів тим, що квантова схема не потребує апроксимації повного основного стану на кожному кроці.

Далі підпросторовий гамільтоніан класично діагоналізується для отримання найменшого власного значення та відповідного власного вектора, що представляють апроксимацію основного стану та його енергії. З покращенням якості підпростору через ітерації обчислений основний стан краще апроксимує істинний основний стан. На цьому етапі може виконуватися додатковий крок відсіювання для видалення з підпростору конфігурацій, що не мають суттєвого внеску в обчислений основний стан. Цей крок забезпечує максимальну компактність підпростору, що переноситься до наступної ітерації. Це оцінюється на основі амплітуд, що повертаються діагоналізацією, оскільки вони представляють важливість внеску кожної конфігурації в обчислений основний стан.

Перевірка збіжності потім визначає, чи подальше навчання покращить результати. Якщо так, виконується необов'язковий крок класичного розширення, параметри квантової схеми оновлюються для подальшої мінімізації обчисленої енергії, і процес повторюється. Крок класичного розширення генерує додаткові конфігурації для підпростору, доповнюючи конфігурації, відібрані з квантового пристрою. Спочатку він визначає конфігурацію з найбільшою амплітудою в результатах діагоналізації, а потім генерує нові конфігурації з одиничними та подвійними збудженнями від визначеної конфігурації. Потрібна кількість цих конфігурацій потім додається до підпростору.

Як тільки визначається, що ітерації збіглись, HI-VQE повертає обчислений основний стан (у формі станів у підпросторі та їхніх амплітуд у хвильовій функції основного стану), його енергію та міру дисперсії енергії, що дає уявлення про те, чи є обчислений стан власним станом гамільтоніана системи.

Користувачі можуть вибирати квантову схему та кількість знімків для кожної квантової схеми, а також контролювати розмір підпростору або вмикати класичну генерацію додаткових конфігурацій на допомогу конфігураціям, згенерованим квантовим способом. Таким чином користувачі можуть налаштовувати поведінку HI-VQE відповідно до своїх цільових застосувань.
## Ліцензування
Зверни увагу, що використання цієї функції Qiskit обмежено задачами, що вимагають не більше 20 кубітів, якщо не отримано ліцензію, яка надає вищий ліміт.

Будь ласка, надішли електронного листа на адресу [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com), якщо хочеш дізнатися про отримання ліцензії.
## Початок роботи
Спочатку [запроси доступ до функції](https://forms.office.com/r/zN3hvMTqJ1).
Потім автентифікуйся за допомогою свого [API-ключа IBM Quantum&reg;](http://quantum.cloud.ibm.com/) і, якщо ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі, вибери функцію Qiskit таким чином:

In [1]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load the function
function = catalog.load("qunova/hivqe-chemistry")

Додаткові параметри можна визначити та надати для молекулярної системи у наступному форматі словника.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Виконай функцію з вхідними даними геометрії та параметрів.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Варто роздрукувати ідентифікатор завдання функції, щоб його можна було надати у запитах на підтримку у разі виникнення проблем.

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

It is a good idea to print the Function job ID so that it can be provided in support requests if something goes wrong.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


Цей приклад використовує 16 кубітів з 8 орбіталями базису sto3g для молекули NH3.
Перевіряй [статус](/guides/functions#check-job-status) свого навантаження функції Qiskit або отримуй [результати](/guides/functions#retrieve-results) таким чином:

In [7]:
print(job.status())

QUEUED


After the job is completed, the results can be obtained with `result()` instance.

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


Після завершення завдання результати можна отримати за допомогою екземпляра `result()`.

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


## Performance

This section shows the demonstrated benchmark calculations of HI-VQE with a 24-qubit case for Li2S, a 40-qubit case for an N2 molecule, and a 44-qubit case for an FeP-NO system.

#### Dissociation potential energy surface curve for an Li2S molecule with 24 qubits

The PES curve is shown with the FCI reference and initial guess from RHF, together with the energy error from the FCI reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

The calculations have been conducted with the following geometries and options.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

The red dots represent the HI-VQE calculation results for six different geometries, and three geometries corresponding to 1.51, 2.40, and 3.80 Angstrom are provided as input in the above cell.

#### Dissociation PES curve for an N2 molecule with 40 qubits

The nitrogen molecule has been identified as a multireference system with large correlation energy contributions beyond the Hartree-Fock state. We conducted a benchmark calculation for the N2 molecule with cc-pvdz basis, (20o,14e) using the homo-lumo active orbital selection. The complete active space (CAS) number to represent this problem is 6,009,350,400. It is not possible to obtain the eigenvalue problem solution (for energy and electronic structure) with this number of states using a powerful desktop (16cpu/64GB). With HI-VQE, users can efficiently search the subspace of CAS states to find chemically accurate results while saving computation resources significantly. The following plots show the PES curve of 40 qubits HI-VQE calculation of the N2 molecule dissociation.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the N2 system](../docs/images/guides/qunova-chemistry/N2_PES_40qubits.avif)

#### Dissociation PES curve for five-coordinated iron(II)-porphyrin with an NO system with 44 qubits

Another interesting chemical system is an iron(II)-porphyrin (FeP) complex with a coordinated nitric oxide (NO) ligand, which represents a biologically relevant metalloporphyrin system that plays crucial roles in various physiological processes. In this example, HI-VQE has been utilized to estimate the accurate potential energy surface curve of the intermolecular interaction between FeP and NO (ground state energy for differently separated geometries). The combined system has 450 orbitals and 202 electrons (450o,202e) with 6-31g(d) basis in total. The homo-lumo active orbital selection was utilized to calculate the smaller case from the real case with (22o,22e). From the following benchmark results, we were able to achieve the chemical accuracy (>&nbsp;1.6 mHa) with a state-of-the-art classical computer chemistry calculation of CASCI(DMRG) (22o,22e) reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the FeP-NO system](../docs/images/guides/qunova-chemistry/fepno_44qubits.avif)

## Benchmarks

- Exact matrix size is the number of determinants for exact solution, such as FCI and CASCI.
- HI-VQE calculation samples and calculates the subspace of it (as in, HI-VQE matrix size).
- Total time includes QPU runtime and Qiskit Function runs with CPU.
- Accuracy is estimated from the energy difference from exact solution.

|   Chemical system  | Number of qubits | Exact matrix size |  HI-VQE matrix size  | E(diff) from exact (mHa) | Number of iteration | Total time    | QPU runtime usage |
| ------------------ | ---------------- | ----------------- | ------------------- | ------------------------ | ------------------- | ------------- | ----------------- |
|  $NH_3$ (8o,10e)   |  16              |  3136             |  1936               |  0.08                    |  6                  | 37 s          | 34 s              |
|  $Li_2S$ (10o,10e) |  20              |  63504            |  3969               |  0.60                    |  5                  | 250 s         | 50 s              |
|  $NH_3$ (15o,10e)  |  30              |  9018009          |  49729              |  0.90                    |  5                  | 354 s         | 54 s              |
|  $N_2$ (16o,14e)   |  32              |  130873600        |  1798281            |  1.10                    |  9                  | 6531 s        | 121 s             |
|  $3H_2O$ (18o,24e) |  36              |  344622096        |  399424             |  0.90                    |  24                 | 5174 s        | 130 s             |
|  $N_2$ (20o,14e)   |  40              |  6009350400       |  9012004            |  1.20                    |  21                 | 46547 s       | 258 s             |

## Fetch error messages

If your workload fails, the status will be `ERROR` and calling `job.result()` will raise an exception:

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Get support

You can send an email to [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) for help with this function.

If you want help with troubleshooting a specific error, please provide the Function job ID of the job that encountered the error.

## Next steps

<Admonition type="tip" title="Recommendations">
- Request access to the function by filling in [this form](https://forms.office.com/r/zN3hvMTqJ1).
- Visit the [API reference](/docs/api/functions/qunova-chemistry) for this Qiskit Function.
- Try the [Compute dissociation PES curve for FeP-NO with HI-VQE](/docs/tutorials/qunova-hivqe) tutorial.
- Review [Pellow-Jarman, A., et al. (2025).  HIVQE: Handover Iterative Variational Quantum Eigensolver for Efficient Quantum Chemistry Calculations. arXiv preprint arXiv:2503.06292](https://arxiv.org/abs/2503.06292).
- Try the [Dissociation PES curves with Qunova HiVQE](/docs/tutorials/qunova-hivqe) tutorial.
</Admonition>